# 01 · Build the medallion

The images are stored as **objects in a governed `raw.images` dataset** (S3, via vended
credentials) — not inside a table. **Bronze** (Iceberg) records the metadata **plus an
`s3://` link** to each raw image; **Silver** captions them; **Gold** embeds them (Lance).
The **`PIPELINE`** service account writes everything through Lakekeeper-vended credentials.

**Prerequisite — pull the models once** (host terminal):
```bash
docker compose exec ollama ollama pull moondream     # vision (Silver)
docker compose exec ollama ollama pull gemma2:2b     # agent chat (notebook 02)
```

In [1]:
import sys, time
sys.path.insert(0, '/work')
import requests
import pyarrow as pa
import mlib, gt
from pylakekeeper import ConflictError
from mlib import (NS_RAW, RAW_TABLE, NS_BRONZE, NS_SILVER, NS_GOLD, GOLD_TABLE, EMB_DIM,
                  VISION_MODEL, OLLAMA_URL, PIPELINE, get_token)
from icehelp import catalog

token = get_token(*PIPELINE)  # pipeline = client-credentials service account
cat = catalog(token)

## 1 · Ingest → `raw.images` (image objects in the lakehouse)

Fetch a few themes from the Met Open Access API and upload each image as an **object** to the
`raw.images` location using vended credentials. `raw.images` is a generic table that serves
as the governed *handle* for the location; the images themselves are plain S3 objects, and
each row keeps its `s3://` URI for Bronze. No local `data/` folder.

In [2]:
API = 'https://collectionapi.metmuseum.org/public/collection/v1'
THEMES = ['cat', 'flower', 'landscape', 'portrait', 'ship', 'horse']
N = 40

def fetch_object(oid):
    r = requests.get(f'{API}/objects/{oid}', timeout=30)
    if r.status_code != 200: return None
    o = r.json()
    if not o.get('isPublicDomain') or not o.get('primaryImageSmall'): return None
    return {'object_id': int(o['objectID']), 'title': o.get('title') or 'Untitled',
            'artist': o.get('artistDisplayName') or '', 'date': o.get('objectDate') or '',
            'medium': o.get('medium') or '', 'department': o.get('department') or '',
            'culture': o.get('culture') or '', 'classification': o.get('classification') or '',
            'image_url': o['primaryImageSmall']}

records, per_theme = {}, max(1, N // len(THEMES) + 2)
for theme in THEMES:
    if len(records) >= N: break
    ids = (requests.get(f'{API}/search', params={'q': theme, 'hasImages': 'true',
           'isPublicDomain': 'true'}, timeout=30).json().get('objectIDs') or [])[:per_theme*4]
    got = 0
    for oid in ids:
        if len(records) >= N or got >= per_theme or oid in records: continue
        obj = fetch_object(oid); time.sleep(0.2)
        if not obj: continue
        try:
            resp = requests.get(obj['image_url'], timeout=60); resp.raise_for_status()
        except requests.RequestException: continue
        obj['image'] = resp.content; obj['theme'] = theme
        records[oid] = obj; got += 1
        print(f'  + [{len(records):>3}] {obj["title"][:50]}')

rows = list(records.values())

# Catalog the raw dataset with pylakekeeper. format='dataset' means Lakekeeper
# governs the location + vends credentials, but we write the objects ourselves
# (it commits no format-specific metadata). Then vend scoped STS creds.
with gt.client(PIPELINE) as c:
    try:
        c.generic_tables.create(NS_RAW, RAW_TABLE, format='dataset', properties={'kind': 'raw-images'})
    except ConflictError:
        pass
    t = c.generic_tables.load(NS_RAW, RAW_TABLE, vended=True)

location = t.location.rstrip('/'); prefix = location.replace('s3://', '')
fs = gt.s3fs(t)  # pyarrow S3 filesystem bound to the vended creds
for r in rows:
    with fs.open_output_stream(f"{prefix}/{r['object_id']}.jpg") as f: f.write(r['image'])
    r['image_uri'] = f"{location}/{r['object_id']}.jpg"
mb = sum(len(r['image']) for r in rows) / 1e6
print(f'uploaded {len(rows)} images ({mb:.1f} MB) as objects under {location}')

  + [  1] The Penitence of Saint Jerome
  + [  2] Stela of the Steward Mentuwoser
  + [  3] Sabine Houdon (1787–1836)
  + [  4] Marie Antoinette in a Park
  + [  5] Young Woman with a Pink
  + [  6] Night-Shining White
  + [  7] Ten Verses on Oxherding
  + [  8] Stirrup-spout bottle with feline and snake
  + [  9] Bowl
  + [ 10] The Fieschi Morgan Staurotheke
  + [ 11] Ia Orana Maria (Hail Mary)
  + [ 12] Settee (one of a pair) (part of a set)
  + [ 13] Poems from the Pavilion of True Meaning
  + [ 14] Virgin and Child with Four Angels
  + [ 15] Clock watch with astronomical dial and sundial
  + [ 16] Damascus Room
  + [ 17] Portrait of Alvise Contarini(?); (verso) A Tethere
  + [ 18] Portrait of a Woman, Possibly a Nun of San Secondo
  + [ 19] Saint Anthony the Abbot in the Wilderness
  + [ 20] Six Jewel Rivers from Various Provinces
  + [ 21] The Forest in Winter at Sunset
  + [ 22] An Egyptian Peasant Woman and Her Child
  + [ 23] "The Concourse of the Birds", Folio 11r from a Man
 

## 2 · Bronze — metadata + `s3://` image link → Iceberg `bronze.artworks`

The queryable metadata plus the `image_uri` pointing at each raw object. Bronze is Iceberg;
the pixels stay in the raw dataset.

In [3]:
META = ['object_id', 'title', 'artist', 'date', 'medium', 'department',
        'culture', 'classification', 'image_url', 'theme']
BRONZE_SCHEMA = pa.schema([(c, pa.int64() if c == 'object_id' else pa.string()) for c in META]
                          + [('image_uri', pa.string())])
cols = {c: [r.get(c) for r in rows] for c in META}
cols['image_uri'] = [r['image_uri'] for r in rows]
bronze_tbl = pa.table(cols, schema=BRONZE_SCHEMA)
BRONZE = f'{NS_BRONZE}.artworks'
if not cat.table_exists(BRONZE): cat.create_table(BRONZE, schema=BRONZE_SCHEMA)
tbl = cat.load_table(BRONZE); tbl.overwrite(bronze_tbl)
print(f'wrote {tbl.scan().to_arrow().num_rows} rows to {BRONZE} (each with an s3:// link to its raw image)')

wrote 40 rows to bronze.artworks (each with an s3:// link to its raw image)


/usr/local/lib/python3.11/site-packages/pyiceberg/table/__init__.py:639: UserWarning: Delete operation did not match any records
  self.delete(


## 3 · Silver — vision captions → Iceberg `silver.artwork_features`

Read each image **object** back from its `image_uri` (vended creds) and caption it with the
local vision model — an attribute the source metadata doesn't have.

In [4]:
import ollama
PROMPT = ('You are cataloguing a museum artwork. In ONE concise sentence, describe what is '
          'visually depicted: subjects, setting, colours and mood. Do not mention the artist '
          'or that it is a painting.')
SILVER_SCHEMA = pa.schema([('object_id', pa.int64()), ('title', pa.string()),
    ('theme', pa.string()), ('caption', pa.string()), ('model', pa.string())])

bronze_rows = cat.load_table(f'{NS_BRONZE}.artworks').scan().to_arrow().to_pylist()
with gt.client(PIPELINE) as c:  # vend creds to read the raw objects
    fs = gt.s3fs(c.generic_tables.load(NS_RAW, RAW_TABLE, vended=True))
oc = ollama.Client(host=OLLAMA_URL)
out = []
for i, r in enumerate(bronze_rows, 1):
    try:
        img = gt.read_object(fs, r['image_uri'])
        caption = ' '.join(oc.generate(model=VISION_MODEL, prompt=PROMPT, images=[img]).get('response', '').split())
    except Exception as e:
        caption = ''; print(f'  ! {r["object_id"]} failed: {e}')
    out.append({'object_id': r['object_id'], 'title': r['title'], 'theme': r['theme'],
                'caption': caption, 'model': VISION_MODEL})
    print(f'  [{i:>3}/{len(bronze_rows)}] {r["title"][:38]!r} -> {caption[:56]!r}')

SILVER = f'{NS_SILVER}.artwork_features'
if not cat.table_exists(SILVER): cat.create_table(SILVER, schema=SILVER_SCHEMA)
tbl = cat.load_table(SILVER); tbl.overwrite(pa.Table.from_pylist(out, schema=SILVER_SCHEMA))
print(f'wrote {tbl.scan().to_arrow().num_rows} rows to {SILVER}')

  [  1/40] 'The Penitence of Saint Jerome' -> 'Three-panel oil on canvas portrait of a man kneeling in '
  [  2/40] 'Stela of the Steward Mentuwoser' -> 'urn with writing on it, people in front of it'
  [  3/40] 'Sabine Houdon (1787–1836)' -> 'urn of a child in a white bust form against a gray backg'
  [  4/40] 'Marie Antoinette in a Park' -> 'urns on a table in front of a tree with leaves.'
  [  5/40] 'Young Woman with a Pink' -> 'urn of flowers on table with woman in red dress holding '
  [  6/40] 'Night-Shining White' -> 'urns with red and white designs on them, one of which ha'
  [  7/40] 'Ten Verses on Oxherding' -> 'urn of water with trees in background, man walking on pa'
  [  8/40] 'Stirrup-spout bottle with feline and s' -> 'urn with dragon head on top and two lions below'
  [  9/40] 'Bowl' -> 'urn with dragon design on side'
  [ 10/40] 'The Fieschi Morgan Staurotheke' -> 'urn with religious figures on it'
  [ 11/40] 'Ia Orana Maria (Hail Mary)' -> '!!!Lorraine Maria!!!'
  [ 1

/usr/local/lib/python3.11/site-packages/pyiceberg/table/__init__.py:639: UserWarning: Delete operation did not match any records
  self.delete(


## 4 · Gold — CLIP embeddings → Lance generic table `gold.image_embeddings`

Read each image object (vended creds), embed with local CLIP, and write the vectors + context
to the Gold **Lance** dataset. This is the asset the agents will want.

In [5]:
import numpy as np, lance, clip_util
from pylakekeeper import GenericTableFormat

silver = {r['object_id']: r['caption'] for r in cat.load_table(f'{NS_SILVER}.artwork_features').scan().to_arrow().to_pylist()}
bronze_rows = cat.load_table(f'{NS_BRONZE}.artworks').scan().to_arrow().to_pylist()
rows = [r for r in bronze_rows if r['object_id'] in silver]
with gt.client(PIPELINE) as c:  # vend creds to read the raw objects
    fs = gt.s3fs(c.generic_tables.load(NS_RAW, RAW_TABLE, vended=True))

vecs = []
for i in range(0, len(rows), 16):
    chunk = rows[i:i+16]
    imgs = [gt.read_object(fs, r['image_uri']) for r in chunk]
    vecs.append(clip_util.embed_images_bytes(imgs))
    print(f'  embedded {min(i+16, len(rows))}/{len(rows)}')
vectors = np.vstack(vecs)

arrow = pa.table({'object_id': pa.array([r['object_id'] for r in rows], pa.int64()),
    'title': pa.array([r['title'] for r in rows]), 'theme': pa.array([r['theme'] for r in rows]),
    'caption': pa.array([silver[r['object_id']] for r in rows]),
    'image_uri': pa.array([r['image_uri'] for r in rows]),
    'vector': pa.array(list(vectors), type=pa.list_(pa.float32(), EMB_DIM))})

# Gold is a real Lance dataset — pylakekeeper catalogs it (format='lance') and
# vends creds; Lance writes the columnar files to the vended location.
with gt.client(PIPELINE) as c:
    try:
        c.generic_tables.create(NS_GOLD, GOLD_TABLE, format=GenericTableFormat.LANCE,
            properties={'embedding-dim': str(EMB_DIM), 'model': clip_util.MODEL_NAME})
    except ConflictError:
        pass
    t = c.generic_tables.load(NS_GOLD, GOLD_TABLE, vended=True)
    lance.write_dataset(arrow, t.location, storage_options=t.lance_storage_options, mode='overwrite')
    t = c.generic_tables.load(NS_GOLD, GOLD_TABLE, vended=True)
    n = lance.dataset(t.location, storage_options=t.lance_storage_options).count_rows()
print(f'wrote {n} embeddings to {NS_GOLD}.{GOLD_TABLE}. Gold is live.')

  embedded 16/40
  embedded 32/40
  embedded 40/40
wrote 40 embeddings to gold.image_embeddings. Gold is live.


[2026-07-22T04:51:29Z WARN  lance::dataset::write::insert] No existing dataset at s3://medallion/warehouse/019f882a-1f03-74b3-965b-34f511007761, it will be created


---
Medallion built. Next: **02-agent-governed.ipynb** — governance in action.